# SatQuery Division 4 — Optical-SAR Cross-Modal Specialist (Official Colab Pipeline)

**Architecture:** Dual ResNet-50 Encoders + Bidirectional Spatial CMAF Fusion Neck + FiLM Query Modulator + 8-Class Task Head  
**Dataset:** Official WHU-OPT-SAR 52-Pair Subset (**17,160 paired 256x256 tiles**)  
**Training Strategy:** Stage 1 (15 Frozen Backbone Epochs) + Stage 2 (5 Top-Layer Fine-Tuning Epochs)  
**Target Classes (8):** Background (0), Farmland (1), City (2), Village (3), Water (4), Forest (5), Road (6), Others (7)  

---
### 8-Stage Execution Workflow
1. **Stage 1 — Runtime & Environment:** CUDA GPU detection and Drive discovery.
2. **Stage 2 — Source Code Synchronization:** Restores verified source tree into `/content/SatQuery` and installs vision dependencies.
3. **Stage 3 — Official Dataset Extraction:** Verifies all 17,160 tiles (11,880 train, 2,310 val, 2,970 test).
4. **Stage 4 — Batch & Dimensional Contract:** Validates 8-class tensor dimensions `[16, 3, 256, 256]`, `[16, 2, 256, 256]`, `[16, 256, 256]`, `[16, 8]`.
5. **Stage 5 — Numerical Stability Audit:** 3-step gradient check with calibrated loss & AMP.
6. **Stage 6 — Official 20-Epoch Training:** Full GPU training run with best checkpoint saving (`cmaf_landcover_best.pth`).
7. **Stage 7 — Final Test Evaluation:** Evaluates held-out test split (2,970 tiles) & 3-way modality ablation (Dual, Optical-only, SAR-only).
8. **Stage 8 — Single Real Inference & Checkpoint Verification:** Computes SHA-256 checksum and produces output artifact.


## Stage 1 — Runtime & GPU Environment
- Verify Python, PyTorch, CUDA, and GPU hardware.
- No external cloud storage or Google Drive required.


In [ ]:
# =============================================================================
# STAGE 1 — Runtime & GPU Environment
# =============================================================================
import os
import sys
from pathlib import Path
import torch

# 1. Set working directory to /content/SatQuery
workspace = Path("/content/SatQuery")
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(str(workspace))
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))

print(f"Working Directory : {os.getcwd()}")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device Name   : {gpu_name}")
    print(f"GPU Total VRAM    : {vram_gb:.2f} GiB")
    print("\n>>> STAGE 1 GPU ENVIRONMENT VERIFIED! <<<")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU")


## Stage 2 — Source Code Synchronization & Dependencies
- Install dependencies (`pydantic-settings`, `timm`, `albumentations`, `tifffile`).
- Forcibly restore verified fresh SatQuery source code.
- Verify critical modules and class weights.


In [ ]:
# =============================================================================
# STAGE 2 — Source Code Synchronization & Dependencies
# =============================================================================
import base64
import hashlib
import io
import os
import shutil
import sys
import site
import zipfile
from pathlib import Path

workspace = Path("/content/SatQuery")
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(str(workspace))

for p in ["/content/SatQuery", str(workspace), os.getcwd()]:
    if p not in sys.path:
        sys.path.insert(0, p)
site.addsitedir(str(workspace))
os.environ["PYTHONPATH"] = f"/content/SatQuery:{os.environ.get('PYTHONPATH', '')}"

print("Installing SatQuery dependencies (pydantic-settings, timm, albumentations, tifffile)...")
!pip install --quiet "pydantic-settings>=2.2.0" "timm" "albumentations" "tifffile"

print("\nForcibly syncing fresh source tree into /content/SatQuery (preserving data/)...\n")
for clean_dir in ["core", "specialists", "registry", "tests"]:
    target_clean = workspace / clean_dir
    if target_clean.exists():
        shutil.rmtree(target_clean)

SOURCE_PAYLOAD_B64 = """..."""

zip_data = base64.b64decode(SOURCE_PAYLOAD_B64)
with zipfile.ZipFile(io.BytesIO(zip_data), "r") as zf:
    zf.extractall(workspace)

print(f"Extracted verified source files to: {workspace}")

from specialists.optical_sar.train_colab import ColabTrainer, DiceLoss, VERSION_MARKER
from specialists.optical_sar.dataset import OpticalSarPairedDataset
from specialists.optical_sar.query_intent import QueryIntentInterpreter, FiLMQueryModulator

print(f"[OK] Loaded ColabTrainer version: {VERSION_MARKER}")
print("\n>>> STAGE 2 SOURCE CODE SYNCHRONIZATION PASSED! <<<")


## Stage 3 — Local Dataset Loading & Verification
- Uses local dataset at `/content/SatQuery/data/official_whu_opt_sar` or direct local file upload.
- Asserts exact split tile counts (11,880 train + 2,310 val + 2,970 test = 17,160 tiles).


In [ ]:
# =============================================================================
# STAGE 3 — Local Dataset Loading & Verification
# =============================================================================
import os
import zipfile
from pathlib import Path

target_dataset_dir = Path("/content/SatQuery/data/official_whu_opt_sar")
target_train_tiles = target_dataset_dir / "train" / "optical"
target_val_tiles = target_dataset_dir / "val" / "optical"
target_test_tiles = target_dataset_dir / "test" / "optical"

train_count = len(list(target_train_tiles.glob("*.png"))) if target_train_tiles.exists() else 0
val_count = len(list(target_val_tiles.glob("*.png"))) if target_val_tiles.exists() else 0
test_count = len(list(target_test_tiles.glob("*.png"))) if target_test_tiles.exists() else 0
total_count = train_count + val_count + test_count

if total_count == 17160 and train_count == 11880 and val_count == 2310 and test_count == 2970:
    print(f"[OK] Local dataset is ALREADY present and verified at: {target_dataset_dir}")
    print(f"  - Train tiles : {train_count:,d} (Expected: 11,880)")
    print(f"  - Val tiles   : {val_count:,d} (Expected: 2,310)")
    print(f"  - Test tiles  : {test_count:,d} (Expected: 2,970)")
    print(f"  - Total tiles : {total_count:,d} (Expected: 17,160)")
else:
    # Check local zip locations first
    dataset_zip = None
    candidate_paths = [
        Path("/content/official_whu_opt_sar_dataset.zip"),
        Path("/content/SatQuery/official_whu_opt_sar_dataset.zip"),
        Path("data/official_whu_opt_sar_dataset.zip"),
        Path("/content/drive/MyDrive/official_whu_opt_sar_dataset.zip"),
    ]
    for cp in candidate_paths:
        if cp.exists() and cp.is_file() and cp.stat().st_size > 100_000_000:
            dataset_zip = cp
            break
            
    if not dataset_zip:
        print("\n[Option] Upload official_whu_opt_sar_dataset.zip directly from your local computer:")
        try:
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded.keys():
                if fname.endswith(".zip"):
                    dataset_zip = Path(fname)
                    break
        except Exception as e:
            print(f"Upload prompt notice: {e}")
            
    if dataset_zip and dataset_zip.exists():
        print(f"\nExtracting local {dataset_zip} ({dataset_zip.stat().st_size / (1024**3):.2f} GiB) into {target_dataset_dir.parent}...")
        target_dataset_dir.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(dataset_zip, "r") as zf:
            zf.extractall(target_dataset_dir.parent)
        print("Extraction complete!")

    train_count = len(list((target_dataset_dir / "train" / "optical").glob("*.png")))
    val_count = len(list((target_dataset_dir / "val" / "optical").glob("*.png")))
    test_count = len(list((target_dataset_dir / "test" / "optical").glob("*.png")))
    total_count = train_count + val_count + test_count

    print(f"\n[Dataset Verification Results]")
    print(f"  - Train tiles : {train_count:,d} (Expected: 11,880)")
    print(f"  - Val tiles   : {val_count:,d} (Expected: 2,310)")
    print(f"  - Test tiles  : {test_count:,d} (Expected: 2,970)")
    print(f"  - Total tiles : {total_count:,d} (Expected: 17,160)")

    assert train_count == 11880, f"Train count mismatch: {train_count}"
    assert val_count == 2310, f"Val count mismatch: {val_count}"
    assert test_count == 2970, f"Test count mismatch: {test_count}"
    print("\n>>> STAGE 3 LOCAL DATASET VERIFICATION PASSED! <<<")


## Stage 4 — Batch & Dimensional Contract
- Load one real training batch (`batch_size=16`).
- Verify dimensions: `optical=[16,3,256,256]`, `sar=[16,2,256,256]`, `label=[16,256,256]`, `intent_vector=[16,8]`.
- Verify FiLM modulation output shape `[16,256,32,32]` and finite values.


In [ ]:
# =============================================================================
# STAGE 4 — Batch & Dimensional Contract
# =============================================================================
import torch
from torch.utils.data import DataLoader
from specialists.optical_sar.dataset import OpticalSarPairedDataset
from specialists.optical_sar.query_intent import QueryIntentInterpreter, FiLMQueryModulator

dataset_path = "data/official_whu_opt_sar"
train_ds = OpticalSarPairedDataset(dataset_path, split="train", num_classes=8)
val_ds = OpticalSarPairedDataset(dataset_path, split="val", num_classes=8)
test_ds = OpticalSarPairedDataset(dataset_path, split="test", num_classes=8)

print("[1/2] Loading single real batch with batch_size=16...")
train_loader = DataLoader(train_ds, batch_size=16, shuffle=False)
batch = next(iter(train_loader))

print("\n[Batch Tensor Dimensions]")
print(f"  - optical       : {list(batch['optical'].shape)} (Expected: [16, 3, 256, 256])")
print(f"  - sar           : {list(batch['sar'].shape)} (Expected: [16, 2, 256, 256])")
print(f"  - label         : {list(batch['label'].shape)} (Expected: [16, 256, 256])")
print(f"  - intent_vector : {list(batch['intent_vector'].shape)} (Expected: [16, 8])")

assert batch['optical'].shape == (16, 3, 256, 256), f"Optical shape mismatch: {batch['optical'].shape}"
assert batch['sar'].shape == (16, 2, 256, 256), f"SAR shape mismatch: {batch['sar'].shape}"
assert batch['label'].shape == (16, 256, 256), f"Label shape mismatch: {batch['label'].shape}"
assert batch['intent_vector'].shape == (16, 8), f"Intent vector shape mismatch: {batch['intent_vector'].shape}"

print("\n[2/2] Testing Query Modulation & FiLM Layer...")
interpreter = QueryIntentInterpreter()
intent_vec = interpreter.get_intent_vector("Identify built-up urban infrastructure and water bodies")
assert intent_vec.shape == (8,), f"Intent vector shape mismatch: {intent_vec.shape}"

film = FiLMQueryModulator(feature_channels=256, num_classes=8)
dummy_feats = torch.randn(16, 256, 32, 32)
mod_feats = film(dummy_feats, batch['intent_vector'])
assert mod_feats.shape == (16, 256, 32, 32), f"FiLM output shape mismatch: {mod_feats.shape}"
assert torch.isfinite(mod_feats).all(), "FiLM output contains non-finite values!"

print(f"  - FiLM output shape : {list(mod_feats.shape)} (Finite: True)")
print("\n>>> STAGE 4 BATCH & DIMENSIONAL CONTRACT PASSED! <<<")


## Stage 5 — Numerical Stability Audit
- Run a 3-step diagnostic with the calibrated loss function.
- Verify all logits, losses, and gradients are finite.


In [ ]:
# =============================================================================
# STAGE 5 — Numerical Stability Audit
# =============================================================================
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from specialists.optical_sar.train_colab import ColabTrainer

print("Initializing ColabTrainer for Numerical Stability Audit...")
trainer = ColabTrainer(dataset_dir="data/official_whu_opt_sar")

print("\n[1/1] Running 3-Step Optimizer Stability Check (Checking Logits, Losses & Gradients)...\n")
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
optimizer = torch.optim.AdamW(
    list(trainer.fusion_neck.parameters()) + list(trainer.task_head.parameters()),
    lr=1e-3,
)

step_count = 0
for batch in train_loader:
    step_count += 1
    opt = batch["optical"].to(trainer.device)
    sar = batch["sar"].to(trainer.device)
    targets = batch["label"].to(trainer.device)
    intent = batch["intent_vector"].to(trainer.device)

    optimizer.zero_grad()
    with torch.amp.autocast("cuda", enabled=trainer.use_amp):
        with torch.no_grad():
            opt_feats = trainer.optical_encoder(opt)
            sar_feats = trainer.sar_encoder(sar)
        fused = trainer.fusion_neck(opt_feats["stride_8"], sar_feats["stride_8"])
        logits, probs = trainer.task_head(fused, intent)
        if logits.shape[2:] != targets.shape[1:]:
            logits = F.interpolate(logits, size=targets.shape[1:], mode="bilinear", align_corners=False)
        ce = trainer.ce_loss_fn(logits.float(), targets)
        dice = trainer.dice_loss_fn(logits.float(), targets)
        loss = ce + 0.5 * dice

    assert torch.isfinite(logits).all(), f"Step {step_count}: Logits contain NaN/Inf!"
    assert torch.isfinite(ce), f"Step {step_count}: CrossEntropy Loss is NaN/Inf!"
    assert torch.isfinite(dice), f"Step {step_count}: Dice Loss is NaN/Inf!"
    assert torch.isfinite(loss), f"Step {step_count}: Total Loss is NaN/Inf ({loss.item()})!"

    if trainer.use_amp:
        trainer.scaler.scale(loss).backward()
        trainer.scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(trainer.fusion_neck.parameters()) + list(trainer.task_head.parameters()), max_norm=1.0)
        trainer.scaler.step(optimizer)
        trainer.scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(trainer.fusion_neck.parameters()) + list(trainer.task_head.parameters()), max_norm=1.0)
        optimizer.step()

    print(f"  Step {step_count:2d}/3 | Batch Loss: {loss.item():.4f} (CE: {ce.item():.4f}, Dice: {dice.item():.4f}) | Gradients: FINITE")
    if step_count >= 3:
        break

print("\n>>> STAGE 5 NUMERICAL STABILITY AUDIT PASSED! <<<")


## Stage 6 — Official 20-Epoch GPU Training Run
- **Stage 1 (Epochs 1-15):** Frozen ResNet-50 Encoders in `.eval()` mode, training CMAF + Task Head with `lr=1e-3`.
- **Stage 2 (Epochs 16-20):** Unfreezes Layer3 of both encoders with `lr=1e-4` for fine-tuning.
- Checkpoint saved to `/content/SatQuery/specialists/optical_sar/checkpoints/cmaf_landcover_best.pth`.


In [ ]:
# =============================================================================
# STAGE 6 — Official 20-Epoch GPU Training Run
# =============================================================================
from specialists.optical_sar.train_colab import ColabTrainer

trainer = ColabTrainer(
    dataset_dir="data/official_whu_opt_sar",
    checkpoint_dir="specialists/optical_sar/checkpoints",
)

best_ckpt_path = trainer.run_colab_training(
    epochs=15,
    fine_tune_epochs=5,
    batch_size=16,
    lr=1e-3,
    num_workers=0,
)

print(f"\n>>> STAGE 6 TRAINING COMPLETE! Best Checkpoint Saved at: {best_ckpt_path} <<<")


## Stage 7 — Final Test Evaluation & Modality Ablation
- Evaluates the best trained checkpoint on the held-out test split (**2,970 tiles**).
- Runs 3-way modality ablation (Dual Optical-SAR vs. Optical-only vs. SAR-only) to quantify cross-modal fusion gain.


In [ ]:
# =============================================================================
# STAGE 7 — Final Test Evaluation & Modality Ablation
# =============================================================================
import torch
from torch.utils.data import DataLoader
from specialists.optical_sar.evaluate import OpticalSarEvaluator
from specialists.optical_sar.dataset import OpticalSarPairedDataset

evaluator = OpticalSarEvaluator(
    checkpoint_path="specialists/optical_sar/checkpoints/cmaf_landcover_best.pth",
    dataset_dir="data/official_whu_opt_sar",
)

test_ds = OpticalSarPairedDataset("data/official_whu_opt_sar", split="test", num_classes=8)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

print("Running Full Evaluation on Held-Out Test Split (2,970 tiles)...")
test_metrics = evaluator.evaluate(test_loader)

print("\n================ TEST SET PERFORMANCE (8-CLASS) ================")
print(f"Pixel Accuracy  : {test_metrics['pixel_acc']*100:.2f}%")
print(f"Mean IoU (mIoU) : {test_metrics['mIoU']:.4f}")
print("----------------------------------------------------------------")
class_names = ["background", "farmland", "city", "village", "water", "forest", "road", "others"]
for k, cname in enumerate(class_names):
    iou = test_metrics.get(f"iou_{cname}", 0.0)
    print(f"  Class {k} ({cname:10s}): IoU = {iou:.4f}")
print("================================================================\n")

print("Running 3-Way Modality Ablation...")
ablation_results = evaluator.run_modality_ablation(test_loader)
print("Ablation Results:", ablation_results)
print("\n>>> STAGE 7 TEST EVALUATION PASSED! <<<")


## Stage 8 — Checkpoint SHA-256 Checksum & Real Inference
- Calculates the exact SHA-256 checksum of `cmaf_landcover_best.pth`.
- Executes 1 real cross-modal inference query and verifies output artifact.


In [ ]:
# =============================================================================
# STAGE 8 — Checkpoint SHA-256 & Real Inference
# =============================================================================
import hashlib
from pathlib import Path
from specialists.optical_sar.specialist import OpticalSarSpecialist
from core.schemas import TaskIntent

ckpt_file = Path("specialists/optical_sar/checkpoints/cmaf_landcover_best.pth")
assert ckpt_file.exists(), f"Checkpoint not found at: {ckpt_file}"

sha256 = hashlib.sha256(ckpt_file.read_bytes()).hexdigest()
size_mb = ckpt_file.stat().st_size / (1024**2)

print("================ CHECKPOINT INTEGRITY REPORT ================")
print(f"Checkpoint Path   : {ckpt_file}")
print(f"File Size         : {size_mb:.2f} MB")
print(f"SHA-256 Checksum  : {sha256}")
print("=============================================================\n")

print("Running Real Multi-Modal Inference Query via OpticalSarSpecialist...")
specialist = OpticalSarSpecialist()
test_opt = "data/official_whu_opt_sar/test/optical/NH49E001013_0_0.png"
test_sar = "data/official_whu_opt_sar/test/sar/NH49E001013_0_0.png"

intent = TaskIntent(
    task_type="optical_sar_cross_modal",
    query_text="Segment urban city infrastructure and water bodies",
    target_classes=["city", "water"],
)

res = specialist.run(optical_image=test_opt, sar_image=test_sar, intent=intent)
print(f"Inference Result Success : {res.success}")
print(f"Confidence Score         : {res.confidence_score:.4f}")
print(f"Artifact Keys            : {list(res.artifacts.keys())}")
print("\n>>> ALL 8 STAGES SUCCESSFULLY COMPLETED! <<<")
